# CS 6220 Homework 1: Iris Classification

## Goal
Predict an Iris flower's class from its sepal length, sepal width,
petal length, and petal width.

## Dataset
This notebook uses `iris.data` downloaded from the UCI Iris dataset.
It contains 150 flowers from three classes.

## Method
1. Load and inspect the data.
2. Split it into 80% training data and 20% test data.
3. Scale the measurements and train a logistic regression model.
4. Measure training time, prediction time, and test accuracy.
5. Inspect the confusion matrix.

In [12]:
import time
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

print("Imports successful!")


Imports successful!


In [13]:
columns = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
    "class",
]

iris = pd.read_csv("iris.data", header=None, names=columns)
iris = iris.dropna(how="all")

print("Dataset shape:", iris.shape)
print(iris["class"].value_counts())
display(iris.head())

Dataset shape: (150, 5)
class
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64


,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [14]:
X = iris.drop(columns="class")
y = iris["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 120
Testing rows: 30


In [15]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000)),
])

train_start = time.perf_counter()
pipeline.fit(X_train, y_train)
training_time = time.perf_counter() - train_start

print(f"Training time: {training_time:.6f} seconds")

Training time: 0.004236 seconds


In [16]:
test_start = time.perf_counter()
predictions = pipeline.predict(X_test)
testing_time = time.perf_counter() - test_start

accuracy = accuracy_score(y_test, predictions)

print(f"Training time: {training_time:.6f} seconds")
print(f"Testing time: {testing_time:.6f} seconds")
print(f"Test accuracy: {accuracy:.2%}")
print(f"Correct predictions: {(predictions == y_test).sum()} out of {len(y_test)}")

Training time: 0.004236 seconds
Testing time: 0.000674 seconds
Test accuracy: 93.33%
Correct predictions: 28 out of 30


In [17]:
matrix = confusion_matrix(
    y_test,
    predictions,
    labels=pipeline.classes_,
)

results = pd.DataFrame(
    matrix,
    index=[f"Actual: {name}" for name in pipeline.classes_],
    columns=[f"Predicted: {name}" for name in pipeline.classes_],
)

display(results)

,Predicted: Iris-setosa,Predicted: Iris-versicolor,Predicted: Iris-virginica
Actual: Iris-setosa,10,0,0
Actual: Iris-versicolor,0,9,1
Actual: Iris-virginica,0,1,9
